#  Frozen SSL feature extraction

Each RAVDESS clip (16 kHz mono) is passed through two frozen pretrained models, **WavLM Base+** and **UniSpeech-SAT Base+**. No training happens here.

All 13 layers are kept and mean-pooled over time:

$$\mathbf{z}_\ell = \frac{1}{T}\sum_{t=1}^{T}\mathbf{h}_{\ell,t}, \quad \ell = 0,\dots,12$$

Output: `wavlm_layers.npy` and `unispeechsat_layers.npy`, shape (1440, 13, 768). Extraction is skipped if the files already exist.

In [ ]:
# just importing the libaries to get our 2 ssl models Unispeech and WavLM
import time
from pathlib import Path
import librosa
import numpy as np
import pandas as pd
import torch
from transformers import AutoFeatureExtractor, UniSpeechSatModel, WavLMModel
# transformers is a library from hugging face that has open access to these models

In [ ]:
# defining csv created in the data preparation step.
meta = pd.read_csv("results/metadata.csv")   
SR = 16000                                   # sample rate for all models

# model spcifications
MODELS = [("microsoft/wavlm-base-plus", WavLMModel, "results/wavlm_layers.npy"),
          ("microsoft/unispeech-sat-base-plus", UniSpeechSatModel, "results/unispeechsat_layers.npy")]

In [ ]:
# loading the data to pass through these 2 pretrained models
print("Clips to process:", len(meta))

# GPU is used if available to speed things up
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# here we put the pretrained model to action to get the ssl features for our ravdess dataset
for model_name, model_class, out_file in MODELS:

    # skiping if already extracted
    if Path(out_file).exists():
        print(out_file, "already exists, skipping")
        continue

    # loading the pretrained model frozen
    extractor = AutoFeatureExtractor.from_pretrained(model_name)
    model = model_class.from_pretrained(model_name).to(device).eval()

    all_clips = []
    for path in meta["path"]:
        audio, _ = librosa.load(path, sr=SR, mono=True)                  # 16 kHz mono
        x = extractor(audio, sampling_rate=SR, return_tensors="pt")["input_values"].to(device)

        with torch.no_grad():                                            # no training required
            out = model(x, output_hidden_states=True)

        # mean-pool each of the 13 layers over time -> (13, 768)
        layers = [h[0].mean(dim=0).cpu().numpy() for h in out.hidden_states]
        all_clips.append(np.stack(layers))

    features = np.stack(all_clips).astype(np.float32)                    # dimension (1440, 13, 768)
    # 1440 rows, 13 layers and 768 featurs per layer
    np.save(out_file, features)
    print("Saved", features.shape, "to", out_file)

In [ ]:
# end of SLL feature extraction